In [1]:
import pandas as pd
import numpy as np

# ---------------------------
# Configuración de parámetros
# ---------------------------
num_houses = 10000
start_date = '2024-05-01'
end_date = '2025-01-07 23:00:00'  # Una semana completa: 7 días * 24h = 168 horas
timestamps = pd.date_range(start=start_date, end=end_date, freq='H')
num_timestamps = len(timestamps)

# ---------------------------
# Creación del DataFrame con producto cartesiano
# ---------------------------
# Generamos los IDs de casa del 1 al 10.000
house_ids = np.arange(1, num_houses + 1)
# Repetimos cada casa para cada timestamp y
# "empaquetamos" los timestamps para cada casa
df = pd.DataFrame({
    'casa_id': np.repeat(house_ids, num_timestamps),
    'timestamp': np.tile(timestamps, num_houses)
})

# ---------------------------
# Simulación de la variable 'temperatura'
# ---------------------------
# Suponemos una temperatura base de 15 grados con una amplitud diaria de 10 grados,
# y añadimos algo de ruido normal (sigma=2).
df['temperatura'] = (15 +
                     10 * np.sin((df['timestamp'].dt.hour - 6) / 24 * 2 * np.pi) +
                     np.random.normal(0, 2, size=len(df)))

# ---------------------------
# Simulación del consumo en kW (consumo_kW)
# ---------------------------
# Para que cada casa tenga una característica única, asignamos un consumo base aleatorio
np.random.seed(42)  # Fijamos la semilla para reproducibilidad
house_baselines = np.random.normal(loc=2, scale=0.5, size=num_houses)  # Media de 2 kW
# Mapeamos este valor a cada fila según el 'casa_id'
baseline_dict = {house: baseline for house, baseline in zip(house_ids, house_baselines)}
df['baseline'] = df['casa_id'].map(baseline_dict)

# Simulamos un efecto diario: las casas tienen mayor consumo en ciertos momentos del día.
# Usamos un patrón senoidal para simular este efecto (valores entre 0 y 1).
df['daily_effect'] = 0.5 * (1 + np.sin((df['timestamp'].dt.hour - 6) / 24 * 2 * np.pi))

# Finalmente, combinamos la información para simular el consumo:
# Se suma la línea base, se añade el efecto diario, se introduce una dependencia (inversa)
# a la desviación de la temperatura respecto a 20°C, y se añade ruido.
df['consumo_kW'] = (df['baseline'] +
                    df['daily_effect'] -
                    0.1 * (df['temperatura'] - 20) +
                    np.random.normal(0, 0.2, size=len(df)))

# Seleccionamos únicamente las columnas relevantes
df = df[['casa_id', 'timestamp', 'temperatura', 'consumo_kW']]

# Visualizamos una muestra del dataset
print("Muestra inicial:")
print(df.head())
print("\nMuestra final:")
print(df.tail())

# Opcional: Guardar el dataset en un CSV (comentado)
df.to_csv("mock_consumo_local.csv", index=False)


C:\Users\alexa\AppData\Local\Temp\ipykernel_37272\3656011418.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  timestamps = pd.date_range(start=start_date, end=end_date, freq='H')


Muestra inicial:
   casa_id           timestamp  temperatura  consumo_kW
0        1 2024-05-01 00:00:00     5.227161    3.589942
1        1 2024-05-01 01:00:00     5.715857    3.632709
2        1 2024-05-01 02:00:00     8.865764    3.309292
3        1 2024-05-01 03:00:00     7.037040    3.713183
4        1 2024-05-01 04:00:00    10.879350    3.649858

Muestra final:
          casa_id           timestamp  temperatura  consumo_kW
60479995    10000 2025-01-07 19:00:00    16.023075    3.328547
60479996    10000 2025-01-07 20:00:00     8.425576    3.639682
60479997    10000 2025-01-07 21:00:00     8.837224    3.433840
60479998    10000 2025-01-07 22:00:00     5.457373    3.792796
60479999    10000 2025-01-07 23:00:00     4.435314    3.644238
